# 🌲 Random Forests
**ISLP Chapter 8 · Data Pattern Recognition for the Rest of Us**

> Random Forests fix the main weakness of single decision trees — high variance — by averaging hundreds of trees, each trained on a random subset of data and features.

### Dataset
**IBM HR Analytics — Employee Attrition**
Predict which employees are likely to leave the company. Real HR data with 35 features including salary, job satisfaction, overtime, and years at company. Directly applicable to workforce planning and retention strategy.

### Time: ~60 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from sklearn.preprocessing import LabelEncoder
print("\u2713 Setup complete")

In [ ]:
# IBM HR Analytics — Employee Attrition dataset
# Source: IBM Watson Analytics sample dataset (widely used in HR analytics)
import numpy as np, pandas as pd

try:
    url = 'https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv'
    attrition = pd.read_csv(url)
    print("Loaded IBM HR dataset from source")
except:
    # Reconstruct synthetically with realistic HR properties
    np.random.seed(42)
    n = 1470
    age = np.random.randint(18, 60, n)
    monthly_income = np.random.lognormal(8.5, 0.5, n).astype(int).clip(1000, 20000)
    job_satisfaction = np.random.randint(1, 5, n)
    overtime = np.random.binomial(1, 0.28, n)
    years_at_company = np.random.randint(0, 35, n)
    work_life_balance = np.random.randint(1, 5, n)
    distance_from_home = np.random.randint(1, 30, n)
    num_companies_worked = np.random.randint(0, 10, n)
    department = np.random.choice(['Sales','R&D','HR'], n, p=[0.3, 0.6, 0.1])

    # Attrition driven by: overtime, low satisfaction, low income, early career
    log_odds = (-1.5
                + 1.2 * overtime
                - 0.3 * job_satisfaction
                - 0.0001 * monthly_income
                + 0.3 * (years_at_company < 3).astype(float)
                + 0.1 * distance_from_home / 10
                + 0.2 * num_companies_worked / 5)
    prob_leave = 1 / (1 + np.exp(-log_odds))
    attrition_flag = (np.random.uniform(0,1,n) < prob_leave).astype(int)

    attrition = pd.DataFrame({
        'Age': age, 'MonthlyIncome': monthly_income,
        'JobSatisfaction': job_satisfaction, 'OverTime': np.where(overtime,'Yes','No'),
        'YearsAtCompany': years_at_company, 'WorkLifeBalance': work_life_balance,
        'DistanceFromHome': distance_from_home, 'NumCompaniesWorked': num_companies_worked,
        'Department': department, 'Attrition': np.where(attrition_flag,'Yes','No')
    })
    print("Using synthetic IBM HR-like dataset")

# Encode target and categoricals
attrition['Attrition_num'] = (attrition['Attrition'] == 'Yes').astype(int)
cat_cols = attrition.select_dtypes('object').columns.drop('Attrition')
for c in cat_cols:
    attrition[c] = LabelEncoder().fit_transform(attrition[c].astype(str))

feature_cols = [c for c in attrition.columns if c not in ['Attrition','Attrition_num']]
X = attrition[feature_cols]
y = attrition['Attrition_num']

print(f"\nIBM HR Attrition: {attrition.shape[0]:,} employees, {len(feature_cols)} features")
print(f"Attrition rate: {y.mean():.1%}  (class imbalance — typical for HR data)")
print(f"\nTop features: {feature_cols[:8]}")

## 🌳 Part 1 — Why Random Forests Beat Single Trees

A single decision tree memorizes the training data (100% training accuracy) but generalizes poorly.
Random Forests train hundreds of trees on bootstrap samples, each considering only a random
subset of features at each split. The decorrelation between trees is what makes the average powerful.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                             random_state=42, stratify=y)

# Compare single tree vs random forest
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_tr, y_tr)

rf = RandomForestClassifier(n_estimators=500, oob_score=True,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

print(f"{'Metric':<28} {'Decision Tree':>14} {'Random Forest':>14}")
print("-" * 58)
for metric, t_val, rf_val in [
    ('Training accuracy',     tree.score(X_tr,y_tr),   rf.score(X_tr,y_tr)),
    ('Test accuracy',         tree.score(X_te,y_te),   rf.score(X_te,y_te)),
    ('Test AUC-ROC',          roc_auc_score(y_te, tree.predict_proba(X_te)[:,1]),
                              roc_auc_score(y_te, rf.predict_proba(X_te)[:,1])),
    ('Generalization gap',    tree.score(X_tr,y_tr)-tree.score(X_te,y_te),
                              rf.score(X_tr,y_tr)-rf.score(X_te,y_te))]:
    print(f"  {metric:<26} {t_val:>14.3f} {rf_val:>14.3f}")
print(f"\n  OOB error (free CV estimate):  {'N/A':>14} {1-rf.oob_score_:>14.3f}")

In [ ]:
# How test error changes with number of trees
n_trees_range = [1, 5, 10, 25, 50, 100, 200, 500]
oob_errors, test_aucs = [], []

for n in n_trees_range:
    rf_n = RandomForestClassifier(n_estimators=n, oob_score=True,
                                   class_weight='balanced', random_state=42, n_jobs=-1)
    rf_n.fit(X_tr, y_tr)
    oob_errors.append(1 - rf_n.oob_score_)
    test_aucs.append(roc_auc_score(y_te, rf_n.predict_proba(X_te)[:,1]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(n_trees_range, oob_errors, 's--', color='#e85d20', lw=2, label='OOB error')
axes[0].set_xlabel('Number of Trees'); axes[0].set_ylabel('Error Rate')
axes[0].set_title('OOB Error Stabilizes Around 100-200 Trees')
axes[0].set_xscale('log'); axes[0].legend()

axes[1].plot(n_trees_range, test_aucs, 'o-', color='#1e5fa8', lw=2, label='Test AUC-ROC')
axes[1].set_xlabel('Number of Trees'); axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('AUC-ROC vs Number of Trees\n(attrition prediction — IBM HR data)')
axes[1].set_xscale('log'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Which factors drive employee attrition? — Variable Importance
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#e85d20' if v > importances.quantile(0.75) else '#1e5fa8'
          for v in importances]
ax.barh(importances.index, importances.values, color=colors, edgecolor='white')
ax.set_xlabel('Mean Decrease in Gini Impurity')
ax.set_title('What Drives Employee Attrition?\n'
             'Random Forest Variable Importance (orange = top quartile)')
plt.tight_layout(); plt.show()

top3 = importances.tail(3).index.tolist()[::-1]
print(f"\n\u2714 Top 3 attrition drivers: {', '.join(top3)}")
print("  These are the levers HR should focus on for retention strategy")

In [ ]:
# ROC curve + business interpretation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

RocCurveDisplay.from_estimator(rf, X_te, y_te, ax=axes[0],
    name=f'Random Forest (AUC={roc_auc_score(y_te,rf.predict_proba(X_te)[:,1]):.3f})')
RocCurveDisplay.from_estimator(tree, X_te, y_te, ax=axes[0],
    name='Single Tree', color='#e85d20', alpha=0.7)
axes[0].set_title('ROC Curve — Attrition Prediction')

# Score distribution by actual outcome
rf_scores = rf.predict_proba(X_te)[:,1]
axes[1].hist(rf_scores[y_te==0], bins=30, alpha=0.6, color='#1e5fa8',
             density=True, label='Stayed')
axes[1].hist(rf_scores[y_te==1], bins=30, alpha=0.6, color='#e85d20',
             density=True, label='Left')
axes[1].axvline(0.5, color='black', lw=1.5, ls='--', label='Default threshold')
axes[1].set_xlabel('Predicted Attrition Probability')
axes[1].set_title('Score Distribution\n(good separation = useful model)')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# @title 📝 Quiz — Random Forests
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is the key difference between bagging and Random Forests?
# @markdown **Q2:** Why does OOB error give a valid estimate of test error?
# @markdown **Q3:** What does variable importance measure in a Random Forest?
# @markdown **Q4:** If HR wants to reduce attrition, which top feature would you investigate first and why?
# @markdown **Q5:** Why does a single decision tree have 100% training accuracy but Random Forest does not?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Random Forests"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*